!pip install numpy==1.23.5 scikit-learn==1.2.2

In [1]:
from google.colab import files
uploaded = files.upload()

for name in uploaded.keys():
    print(f"{name} uploaded successfully")

Saving model.pkl to model.pkl
model.pkl uploaded successfully


In [2]:
from google.colab import files
uploaded = files.upload()

for name in uploaded.keys():
    print(f"{name} uploaded successfully")

Saving vectorizer.pkl to vectorizer.pkl
vectorizer.pkl uploaded successfully


In [3]:
from google.colab import files
uploaded = files.upload()

for name in uploaded.keys():
    print(f"{name} uploaded successfully")

Saving balanced_edufeed_dataset.xlsx to balanced_edufeed_dataset.xlsx
balanced_edufeed_dataset.xlsx uploaded successfully


In [4]:
from google.colab import files
uploaded = files.upload()

for name in uploaded.keys():
    print(f"{name} uploaded successfully")

Saving edufeed_zenml_groq.ipynb to edufeed_zenml_groq.ipynb
edufeed_zenml_groq.ipynb uploaded successfully


In [5]:
import joblib

model = joblib.load("model.pkl")
vectorizer = joblib.load("vectorizer.pkl")

print("All files loaded successfully ")

All files loaded successfully 


In [6]:
print("Pipeline Stages:")
stages = ["Ingest", "Anonymise", "Embed", "FAISS", "Groq Insights", "Export"]

for i, stage in enumerate(stages, 1):
    print(f"Step {i}: {stage}")

Pipeline Stages:
Step 1: Ingest
Step 2: Anonymise
Step 3: Embed
Step 4: FAISS
Step 5: Groq Insights
Step 6: Export


In [7]:
import pandas as pd

df = pd.read_excel("balanced_edufeed_dataset.xlsx")

print("Total rows:", len(df))
df.head()

Total rows: 35421


,professor_name,school_name,department_name,local_name,state_name,year_since_first_review,star_rating,take_again,diff_index,tag_professor,...,post_month,date_missing_flag,grade_group,help_ratio,difficulty_category,experience_bucket,comment_length_cat,tag_count,course_mode,sentiment_label
0,Leslie Looney,University Of Illinois At Urbana-Champaign,Astronomy Department,Champaign\Xe2\X80\X93Urbana,Il,11,4.7,0.75,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,...,6.0,0,B Range,0.0,Moderate,10+ yrs,Medium,3,Offline,Positive
1,Leslie Looney,University Of Illinois At Urbana-Champaign,Astronomy Department,Champaign\Xe2\X80\X93Urbana,Il,11,4.7,0.75,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,...,4.0,0,A Range,0.0,Easy,10+ yrs,Medium,3,Offline,Positive
2,Leslie Looney,University Of Illinois At Urbana-Champaign,Astronomy Department,Champaign\Xe2\X80\X93Urbana,Il,11,4.7,0.75,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,...,7.0,0,Not Reported,0.0,Moderate,10+ yrs,Medium,3,Offline,Positive
3,Leslie Looney,University Of Illinois At Urbana-Champaign,Astronomy Department,Champaign\Xe2\X80\X93Urbana,Il,11,4.7,0.75,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,...,8.0,0,A Range,0.0,Moderate,10+ yrs,Long,3,Offline,Positive
4,Leslie Looney,University Of Illinois At Urbana-Champaign,Astronomy Department,Champaign\Xe2\X80\X93Urbana,Il,11,4.7,0.75,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,...,2.0,0,Not Reported,0.0,Easy,10+ yrs,Long,3,Offline,Positive


In [8]:
print(df.isnull().sum())

professor_name        0
school_name           0
department_name       0
local_name            0
state_name            0
                     ..
experience_bucket     0
comment_length_cat    0
tag_count             0
course_mode           0
sentiment_label       0
Length: 62, dtype: int64


In [9]:
print(df.columns)

Index(['professor_name', 'school_name', 'department_name', 'local_name',
       'state_name', 'year_since_first_review', 'star_rating', 'take_again',
       'diff_index', 'tag_professor', 'num_student', 'post_date',
       'name_onlines', 'name_not_onlines', 'student_star', 'student_difficult',
       'attence', 'for_credits', 'would_take_agains', 'grades', 'help_useful',
       'help_not_useful', 'comments', 'word_comment', 'gender', 'race',
       'asian', 'hispanic', 'nh_black', 'nh_white', 'gives_good_feedback',
       'caring', 'respected', 'participation_matters',
       'clear_grading_criteria', 'skip_class', 'amazing_lectures',
       'inspirational', 'tough_grader', 'hilarious', 'get_ready_to_read',
       'lots_of_homework', 'accessible_outside_class', 'lecture_heavy',
       'extra_credit', 'graded_by_few_things', 'group_projects', 'test_heavy',
       'so_many_papers', 'beware_of_pop_quizzes', 'IsCourseOnline',
       'post_year', 'post_month', 'date_missing_flag', 'grade_g

In [10]:
import re
import pandas as pd

def contains_sensitive(text):
    if pd.isna(text):
        return False

    email = re.search(r'\S+@\S+', str(text))
    numbers = re.search(r'\d{10}', str(text))

    return bool(email or numbers)

In [11]:
df["sensitive_flag"] = df["comments"].apply(contains_sensitive)

In [12]:
col = df.columns[0]  # or manually set correct name

df["sensitive_flag"] = df[col].apply(contains_sensitive)

print("Sensitive data found:", df["sensitive_flag"].sum())

Sensitive data found: 0


In [13]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.8 MB/s eta 0:00:00


In [14]:
import logging
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

In [15]:
from sentence_transformers import SentenceTransformer
model_embed = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [16]:
texts = df["comments"].astype(str).tolist()

In [17]:
import time

start = time.time()

embeddings = model_embed.encode(texts[:500])  # use sample first

end = time.time()

print("Embedding time:", end - start)

Embedding time: 16.336356163024902


In [18]:
!pip install -q faiss-cpu

In [19]:
import faiss
import numpy as np

dimension = len(embeddings[0])

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("FAISS index created with", index.ntotal, "vectors")

FAISS index created with 500 vectors


In [20]:
query = embeddings[0].reshape(1, -1)

start = time.time()
D, I = index.search(query, k=5)
end = time.time()

print("Search time:", end - start)

Search time: 0.022634029388427734


In [21]:
import json

results = {
    "sample_results": I.tolist(),
    "distances": D.tolist()
}

with open("results.json", "w") as f:
    json.dump(results, f)

print("Results exported")

Results exported


In [22]:
!ls

balanced_edufeed_dataset.xlsx  model.pkl     sample_data
edufeed_zenml_groq.ipynb       results.json  vectorizer.pkl


In [23]:
import json

with open("results.json", "r") as f:
    data = json.load(f)

print("Export verified ")

Export verified 


In [24]:
df.head()
vectorizer
model

LogisticRegression(max_iter=1000)

In [28]:
feature_names = vectorizer.get_feature_names_out()
print(f"Total features:",{len(feature_names)})

Total features: {10000}


In [26]:
important_words = ["hard", "helpful", "easy", "difficult", "pass"]

for word in important_words:
    if word in feature_names:
        print(f"{word} is in vocabulary")
    else:
        print(f"{word} NOT found")

hard is in vocabulary
helpful is in vocabulary
easy is in vocabulary
difficult is in vocabulary
pass is in vocabulary


In [27]:
print(df.columns)

Index(['professor_name', 'school_name', 'department_name', 'local_name',
       'state_name', 'year_since_first_review', 'star_rating', 'take_again',
       'diff_index', 'tag_professor', 'num_student', 'post_date',
       'name_onlines', 'name_not_onlines', 'student_star', 'student_difficult',
       'attence', 'for_credits', 'would_take_agains', 'grades', 'help_useful',
       'help_not_useful', 'comments', 'word_comment', 'gender', 'race',
       'asian', 'hispanic', 'nh_black', 'nh_white', 'gives_good_feedback',
       'caring', 'respected', 'participation_matters',
       'clear_grading_criteria', 'skip_class', 'amazing_lectures',
       'inspirational', 'tough_grader', 'hilarious', 'get_ready_to_read',
       'lots_of_homework', 'accessible_outside_class', 'lecture_heavy',
       'extra_credit', 'graded_by_few_things', 'group_projects', 'test_heavy',
       'so_many_papers', 'beware_of_pop_quizzes', 'IsCourseOnline',
       'post_year', 'post_month', 'date_missing_flag', 'grade_g

In [29]:
df["comments"].isna().sum()

np.int64(4)

In [30]:
df["comments"] = df["comments"].fillna("")

In [31]:
df["comments"] = df["comments"].astype(str)

In [32]:
feature_names = vectorizer.get_feature_names_out()

X = vectorizer.transform(df["comments"])

import numpy as np
scores = np.asarray(X.mean(axis=0)).flatten()

top_indices = scores.argsort()[-10:][::-1]

print("Top important words:\n")
for i in top_indices:
    print(feature_names[i])

Top important words:

the
and
you
to
he
is
class
she
comments
no


In [35]:
# Ensure text is clean (IMPORTANT)
df["comments"] = df["comments"].fillna("").astype(str)

# Transform text to features
X = vectorizer.transform(df["comments"])

# True labels
y_true = df["sentiment_label"]

# Predictions from Logistic Regression model
y_pred = model.predict(X)

In [36]:
from sklearn.metrics import classification_report

print("Classification Report:\n")
print(classification_report(y_true, y_pred, digits=2))

Classification Report:

              precision    recall  f1-score   support

    Negative       0.83      0.90      0.86     11807
     Neutral       0.81      0.82      0.81     11807
    Positive       0.88      0.79      0.84     11807

    accuracy                           0.84     35421
   macro avg       0.84      0.84      0.84     35421
weighted avg       0.84      0.84      0.84     35421



In [37]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[10574   917   316]
 [ 1166  9713   928]
 [ 1010  1430  9367]]


In [38]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_true, y_pred)
print(f"Accuracy:{accuracy:.2f}")

Accuracy:0.84


In [39]:
!pip install -q transformers

In [40]:
from transformers import pipeline

In [41]:
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

In [42]:
classifier = pipeline("zero-shot-classification", model="microsoft/deberta-v3-base")

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [43]:
labels = ["positive", "neutral", "negative"]

In [44]:
test_sentences = [
    "The course is not bad at all",
    "Assignments are killing me but I learned a lot",
    "Yeah great teaching... not really",
    "The faculty is okay, nothing special"
]

for text in test_sentences:
    result = classifier(text, candidate_labels=labels)
    print("\nText:", text)
    print("Prediction:", result["labels"][0])


Text: The course is not bad at all
Prediction: negative

Text: Assignments are killing me but I learned a lot
Prediction: positive

Text: Yeah great teaching... not really
Prediction: negative

Text: The faculty is okay, nothing special
Prediction: neutral


In [45]:
results = []

sample_texts = df["comments"].sample(5)

for text in sample_texts:
    lr_pred = model.predict(vectorizer.transform([text]))[0]
    bert_pred = classifier(text, candidate_labels=labels)["labels"][0]

    results.append((text, lr_pred, bert_pred))

In [46]:
for text, lr, bert in results:
    print("\nText:", text[:100])
    print("Logistic Regression:", lr)
    print("DeBERTa:", bert)


Text: Very helpful and nice, especially at HW help sessions and office hours. If you\'re into physics and 
Logistic Regression: Neutral
DeBERTa: positive

Text: Great teacher
Logistic Regression: Positive
DeBERTa: positive

Text: he sucks if you have any class with him drop out fast
Logistic Regression: Negative
DeBERTa: negative

Text: Dr. Frank is a great professor! His lectures tend to be on the boring side but he tries to make it e
Logistic Regression: Positive
DeBERTa: positive

Text: Class is INTENSE! Omg I had to work my butt off for an A. Lots of papers to write & her directions a
Logistic Regression: Neutral
DeBERTa: neutral


In [47]:
mismatch = [r for r in results if r[1] != r[2]]

print("\nTotal mismatches:", len(mismatch))


Total mismatches: 5


In [48]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

    Negative       0.83      0.90      0.86     11807
     Neutral       0.81      0.82      0.81     11807
    Positive       0.88      0.79      0.84     11807

    accuracy                           0.84     35421
   macro avg       0.84      0.84      0.84     35421
weighted avg       0.84      0.84      0.84     35421



In [49]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_true, y_pred))

Accuracy: 0.8371869794754524
